# Project: Comparing General vs Biomedical BERT for ICD Classification

## Experiments
1. BERT-base + Cross Entropy
2. BiomedBERT + Cross Entropy
3. BiomedBERT + Weighted Cross Entropy

## Libraries

In [1]:
import os
import sys
import json
import random
from typing import Dict, List

import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, f1_score, top_k_accuracy_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)

# If needed, install once:
# !pip install -q transformers[torch] datasets evaluate accelerate scikit-learn pandas numpy

## Basic setup

In [2]:
# Basic Env.
print("=== Basic Environment Check ===")
print("Python executable:", sys.executable)
print("Current working directory:", os.getcwd())

# CUDA
print("\n=== PyTorch / CUDA Check ===")
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    x = torch.tensor([1.0, 2.0, 3.0]).cuda()
    print("GPU tensor device:", x.device)
else:
    print("⚠️ CUDA not detected in this notebook kernel")

# seed
SEED = 359
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# directory
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, 'data')
MODEL_DIR = os.path.join(BASE_DIR, 'models')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')

#
OUTPUT_DIR = os.path.join(OUTPUT_DIR, "base_bert_ce")
MODEL_DIR = os.path.join(MODEL_DIR, "base_bert_ce")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

LABEL2ID_PATH = DATA_DIR + "/label2id_top50.json"
ID2LABEL_PATH = DATA_DIR + "/id2label_top50.json"

TRAIN_PATH = os.path.join(DATA_DIR, 'train_50_19803.csv')
VAL_PATH = os.path.join(DATA_DIR, 'val_50_2475.csv')
TEST_PATH = os.path.join(DATA_DIR, 'test_50_2476.csv')
print("\n=== Dataset ===")
print(TRAIN_PATH)
print(VAL_PATH)
print(TEST_PATH)

# dataset
TEXT_COLUMN = "filtered_text"
LABEL_COLUMN = "labels"

# model
MODEL_CHECKPOINT = "google-bert/bert-base-uncased"
MAX_LENGTH = 256
print("\n=== Model ===")
print(MODEL_CHECKPOINT)

=== Basic Environment Check ===
Python executable: c:\Users\Minkyu Ham\Desktop\stat359\.venv\Scripts\python.exe
Current working directory: c:\Users\Minkyu Ham\Desktop\stat359\student\final_project\clinical_note_icd_llm\sub_experiment

=== PyTorch / CUDA Check ===
Torch version: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
GPU tensor device: cuda:0

=== Dataset ===
c:\Users\Minkyu Ham\Desktop\stat359\student\final_project\clinical_note_icd_llm\sub_experiment\data\train_50_19803.csv
c:\Users\Minkyu Ham\Desktop\stat359\student\final_project\clinical_note_icd_llm\sub_experiment\data\val_50_2475.csv
c:\Users\Minkyu Ham\Desktop\stat359\student\final_project\clinical_note_icd_llm\sub_experiment\data\test_50_2476.csv

=== Model ===
google-bert/bert-base-uncased


## Dataset

### Load label mapping

In [3]:
with open(LABEL2ID_PATH, "r", encoding="utf-8") as f:
    label2id = json.load(f)
with open(ID2LABEL_PATH, "r", encoding="utf-8") as f:
    id2label_raw = json.load(f)

# JSON keys may come in as strings
id2label = {int(k): v for k, v in id2label_raw.items()}
num_labels = len(label2id)

print("===== LABEL INFO =====")
print("num_labels:", num_labels)
print("sample label2id:", list(label2id.items())[:5])

===== LABEL INFO =====
num_labels: 50
sample label2id: [('A41', 0), ('C34', 1), ('C79', 2), ('E10', 3), ('E11', 4)]


### Load CSV data files

In [4]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

train_df = train_df[["filtered_text", "labels", "major_code"]].copy()
val_df = val_df[["filtered_text", "labels", "major_code"]].copy()
test_df = test_df[["filtered_text", "labels", "major_code"]].copy()

print("===== DATA CHECK =====")
print("text column:", TEXT_COLUMN)
print("train shape:", train_df.shape)
print("val shape:", val_df.shape)
print("test shape:", test_df.shape)

print("\nSample training example:\n```")
print(train_df.iloc[0][TEXT_COLUMN][:300])
print("```\nmajor:", train_df.iloc[0]['major_code'])
label_id = train_df.iloc[0][LABEL_COLUMN]
print("label:", label_id, "==", id2label[label_id])

===== DATA CHECK =====
text column: filtered_text
train shape: (19803, 3)
val shape: (2475, 3)
test shape: (2476, 3)

Sample training example:
```
Chest pain, hand, foot, and back pain following assault.

Patient is a [Age] year old male with a history of alcoholism, dyslipidemia, diabetes, and hypertension, brought to the ED via ambulance after a physical assault and loss of consciousness. He presented with complaints of chest pain, nausea/vo
```
major: R07
label: 39 == R07


### Convert to Hugging Face Datasets

In [5]:
dataset_dict = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(val_df, preserve_index=False),
    "test": Dataset.from_pandas(test_df, preserve_index=False),
})
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['filtered_text', 'labels', 'major_code'],
        num_rows: 19803
    })
    validation: Dataset({
        features: ['filtered_text', 'labels', 'major_code'],
        num_rows: 2475
    })
    test: Dataset({
        features: ['filtered_text', 'labels', 'major_code'],
        num_rows: 2476
    })
})

## Tokenizer

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
tokenizer

BertTokenizerFast(name_or_path='google-bert/bert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [7]:
def tokenize_batch(batch: Dict[str, List[str]]) -> Dict[str, List[List[int]]]:
    return tokenizer(
        batch[TEXT_COLUMN],
        truncation=True,
        max_length=MAX_LENGTH,  # 256
        padding=False,  # for dynamic padding
    )

tokenized_datasets = dataset_dict.map(
    tokenize_batch,
    batched=True,
    desc="Tokenizing",
)

print("===== TOKENIZED DATASET =====")
print(tokenized_datasets)
print(tokenized_datasets["train"][0].keys())

# Dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Tokenizing:   0%|          | 0/19803 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/2475 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/2476 [00:00<?, ? examples/s]

===== TOKENIZED DATASET =====
DatasetDict({
    train: Dataset({
        features: ['filtered_text', 'labels', 'major_code', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 19803
    })
    validation: Dataset({
        features: ['filtered_text', 'labels', 'major_code', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2475
    })
    test: Dataset({
        features: ['filtered_text', 'labels', 'major_code', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2476
    })
})
dict_keys(['filtered_text', 'labels', 'major_code', 'input_ids', 'token_type_ids', 'attention_mask'])


In [8]:
# tokenizer check
sample_text = train_df['filtered_text'][0]
sample_ids = tokenizer(sample_text, add_special_tokens=True)["input_ids"]
print("===== TOKENIZER CHECK =====")
print("sample text:\n```\n", sample_text)
print("```\nencoded ids:", sample_ids)
print("decoded:", tokenizer.decode(sample_ids))
print("token length:", len(sample_ids))

===== TOKENIZER CHECK =====
sample text:
```
 Chest pain, hand, foot, and back pain following assault.

Patient is a [Age] year old male with a history of alcoholism, dyslipidemia, diabetes, and hypertension, brought to the ED via ambulance after a physical assault and loss of consciousness. He presented with complaints of chest pain, nausea/vomiting/diarrhea, pelvic and abdominal pain. He endorsed cough, fever, chills, night sweats, weight loss, headache, congestion, nausea, diarrhea, increased urinary frequency/volume, and dysuria for 1 month. He reports a 6-month history of exertional chest tightness, palpitations, and paroxysmal nocturnal dyspnea/chest pain relieved by sitting up. He also reports bilateral foot numbness up to the ankles. He was assaulted the night prior to admission.

IV fluids (1L NS), multivitamins, KCl, Aspirin, Oxycodone, Diazepam (for CIWA >10), Thiamine, Folic Acid. Discharged on Aspirin, Atorvastatin, Gabapentin, Acetaminophen, Folic Acid, Thiamine, Multivit

## Load model

In [9]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)
model

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

## Metrics

In [10]:
# accuracy_metric = evaluate.load("accuracy")
# f1_metric = evaluate.load("f1")
# use sklearn instead
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.from_numpy(logits), dim=-1).numpy()  # Hit@k

    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")

    # Hit@3 / Hit@5
    hit3 = top_k_accuracy_score(labels, probs, k=3, labels=np.arange(num_labels))
    hit5 = top_k_accuracy_score(labels, probs, k=5, labels=np.arange(num_labels))

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "hit@3": hit3,
        "hit@5": hit5,
    }

## Training

### Training arguments: training rules
- Local RTX 4050 laptop-friendly setting

In [11]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=20,
    logging_first_step=True,

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,

    num_train_epochs=5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    lr_scheduler_type="linear",  # stable for BERTs
    max_grad_norm=1.0,

    fp16=torch.cuda.is_available(),

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    save_total_limit=2,

    report_to="none",
    seed=SEED,
)
training_args

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False

### Trainer

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer

C:\Users\Minkyu Ham\AppData\Local\Temp\ipykernel_23972\2728132684.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


### Train

In [13]:
print(tokenized_datasets["train"][0]["labels"])
print(len(tokenized_datasets["train"][0]["input_ids"]))

39
256


In [14]:
print("===== START TRAINING =====")
train_result = trainer.train()
print("===== TRAIN FINISHED =====")

# Save final model + tokenizer
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

===== START TRAINING =====


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Hit@3,Hit@5
1,1.220000,1.243921,0.675960,0.626791,0.857778,0.911111
2,0.963700,0.933161,0.733737,0.714580,0.912727,0.949495
3,0.662600,0.884353,0.755152,0.750056,0.923636,0.951919
4,0.594600,0.926211,0.766061,0.759983,0.921616,0.949091
5,0.259900,0.957872,0.767273,0.762069,0.920000,0.947879


===== TRAIN FINISHED =====


('c:\\Users\\Minkyu Ham\\Desktop\\stat359\\student\\final_project\\clinical_note_icd_llm\\sub_experiment\\models\\base_bert_ce\\tokenizer_config.json',
 'c:\\Users\\Minkyu Ham\\Desktop\\stat359\\student\\final_project\\clinical_note_icd_llm\\sub_experiment\\models\\base_bert_ce\\special_tokens_map.json',
 'c:\\Users\\Minkyu Ham\\Desktop\\stat359\\student\\final_project\\clinical_note_icd_llm\\sub_experiment\\models\\base_bert_ce\\vocab.txt',
 'c:\\Users\\Minkyu Ham\\Desktop\\stat359\\student\\final_project\\clinical_note_icd_llm\\sub_experiment\\models\\base_bert_ce\\added_tokens.json',
 'c:\\Users\\Minkyu Ham\\Desktop\\stat359\\student\\final_project\\clinical_note_icd_llm\\sub_experiment\\models\\base_bert_ce\\tokenizer.json')

## Evaluation

In [15]:
print("===== VALIDATION EVALUATION =====")
val_metrics = trainer.evaluate(eval_dataset=tokenized_datasets["validation"])
val_metrics

===== VALIDATION EVALUATION =====


{'eval_loss': 0.9578717947006226,
 'eval_accuracy': 0.7672727272727272,
 'eval_macro_f1': 0.762069249231746,
 'eval_hit@3': 0.92,
 'eval_hit@5': 0.9478787878787879,
 'eval_runtime': 11.7319,
 'eval_samples_per_second': 210.962,
 'eval_steps_per_second': 13.212,
 'epoch': 5.0}

In [16]:
print("\n===== TEST EVALUATION =====")
test_metrics = trainer.evaluate(eval_dataset=tokenized_datasets["test"])
test_metrics


===== TEST EVALUATION =====


{'eval_loss': 0.945213794708252,
 'eval_accuracy': 0.768578352180937,
 'eval_macro_f1': 0.7654549322542633,
 'eval_hit@3': 0.9168012924071083,
 'eval_hit@5': 0.9547657512116317,
 'eval_runtime': 11.6345,
 'eval_samples_per_second': 212.816,
 'eval_steps_per_second': 13.322,
 'epoch': 5.0}

In [17]:
# Helper for JSON safe conversion
def convert_metrics(metrics):
    return {k: float(v) for k, v in metrics.items()}

# Save metrics
val_metrics_path = os.path.join(OUTPUT_DIR, "validation_metrics.json")
test_metrics_path = os.path.join(OUTPUT_DIR, "test_metrics.json")

with open(val_metrics_path, "w", encoding="utf-8") as f:
    json.dump(convert_metrics(val_metrics), f, indent=2)

with open(test_metrics_path, "w", encoding="utf-8") as f:
    json.dump(convert_metrics(test_metrics), f, indent=2)

print("===== METRICS SAVED =====")
print("Val metrics saved to:", val_metrics_path)
print("Test metrics saved to:", test_metrics_path)

===== METRICS SAVED =====
Val metrics saved to: c:\Users\Minkyu Ham\Desktop\stat359\student\final_project\clinical_note_icd_llm\sub_experiment\outputs\base_bert_ce\validation_metrics.json
Test metrics saved to: c:\Users\Minkyu Ham\Desktop\stat359\student\final_project\clinical_note_icd_llm\sub_experiment\outputs\base_bert_ce\test_metrics.json


In [18]:
# Save trainer log history
log_history = trainer.state.log_history
log_history_path = os.path.join(OUTPUT_DIR, "trainer_log_history.json")

with open(log_history_path, "w", encoding="utf-8") as f:
    json.dump(log_history, f, indent=2)

print("Trainer log history saved to:", log_history_path)

Trainer log history saved to: c:\Users\Minkyu Ham\Desktop\stat359\student\final_project\clinical_note_icd_llm\sub_experiment\outputs\base_bert_ce\trainer_log_history.json


In [19]:
# Save train metrics
train_metrics_path = os.path.join(OUTPUT_DIR, "train_metrics.json")
train_metrics = train_result.metrics if hasattr(train_result, "metrics") else {}

with open(train_metrics_path, "w", encoding="utf-8") as f:
    json.dump(convert_metrics(train_metrics), f, indent=2)

print("Train metrics saved to:", train_metrics_path)

Train metrics saved to: c:\Users\Minkyu Ham\Desktop\stat359\student\final_project\clinical_note_icd_llm\sub_experiment\outputs\base_bert_ce\train_metrics.json


## Simple inference example

In [20]:
print("===== INFERENCE EXAMPLE =====")
sample_note = test_df.iloc[0][TEXT_COLUMN]

inputs = tokenizer(
    sample_note,
    return_tensors="pt",
    truncation=True,
    max_length=MAX_LENGTH,
)

if torch.cuda.is_available():
    inputs = {k: v.cuda() for k, v in inputs.items()}
    trainer.model.cuda()

trainer.model.eval()
with torch.no_grad():
    outputs = trainer.model(**inputs)
    logits = outputs.logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

pred_id = int(np.argmax(probs))
top5_ids = np.argsort(probs)[::-1][:5]

print("Predicted label id:", pred_id)
print("Predicted ICD code:", id2label[pred_id])
print("Top-5 predictions:")
for idx in top5_ids:
    print(f"  {idx}: {id2label[int(idx)]} (prob={probs[idx]:.4f})")

===== INFERENCE EXAMPLE =====
Predicted label id: 22
Predicted ICD code: J44
Top-5 predictions:
  22: J44 (prob=0.9591)
  23: J96 (prob=0.0200)
  21: J18 (prob=0.0032)
  24: K35 (prob=0.0012)
  9: I13 (prob=0.0012)
